In [41]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
# Reading of the csv file into a dataframe
df = pd.read_csv("chickwts.csv")                 

In [43]:
# EDA start
print(f"Data Shape: {df.shape}")
df.head()


Data Shape: (71, 3)


,rownames,weight,feed
0,1,179,horsebean
1,2,160,horsebean
2,3,136,horsebean
3,4,227,horsebean
4,5,217,horsebean


In [63]:
# Checking for missing and duplicate values
print(f"Missing values in the dataset: {df.isna().sum().sum()}")
print(f"Duplicated values in the dataset: {df.duplicated().sum()}")

Missing values in the dataset: 0
Duplicated values in the dataset: 0


In [45]:
# Doing an aggregate of chick-level data into feed-level profiles
feed_profiles = df.groupby('feed')['weight'].agg(
    mean_weight='mean',
    median_weight='median',
    std_weight='std',
    min_weight='min',
    max_weight='max',
    count='count'
).reset_index()
print(feed_profiles)

        feed  mean_weight  median_weight  std_weight  min_weight  max_weight  \
0     casein   323.583333          342.0   64.433840         216         404   
1  horsebean   160.200000          151.5   38.625841         108         227   
2    linseed   218.750000          221.0   52.235698         141         309   
3   meatmeal   276.909091          263.0   64.900623         153         380   
4    soybean   246.428571          248.0   54.129068         158         329   
5  sunflower   328.916667          328.0   48.836384         226         423   

   count  
0     12  
1     10  
2     12  
3     11  
4     14  
5     12  


In [50]:
# Separating feed names from the numeric features
feed_labels = feed_profiles['feed']
# Drops the feed column to leave only the numeric features
X_feed = feed_profiles.drop('feed', axis=1)



In [35]:
# Standardizing
scaler = StandardScaler()
X_feed_scaled = scaler.fit_transform(X_feed)

In [36]:
# Applies PCA to reduce data to one component
pca = PCA(n_components=1)
X_feed_pca = pca.fit_transform(X_feed_scaled)

In [60]:
# Computes cosine similarity
sim_matrix = cosine_similarity(X_feed_pca)
sim_df = pd.DataFrame(sim_matrix, index=feed_labels, columns=feed_labels)

def recommend_feed(feed_name):
    scores = sim_df[feed_name].drop(feed_name)
    best_feed = scores.idxmax()
    best_score = scores.max()
    return best_feed, best_score
    
feed_queried = 'horsebean'
feed, score = recommend_feed(feed_queried)
print(f"Recommended feed for {feed_queried}: {feed} (similarity: {score:.2f})")    


Recommended feed for horsebean: linseed (similarity: 1.00)


THE OBJECTIVE AND APPROACH
This project builds a content-based recommendation system to help an agricultural supply company suggest comparable feed types to farmers, using the classic chicken weights dataset as a proxy for real product-performance data. Rather than comparing individual chicks, the raw chick-level weight records were aggregated into one profile per feed type, summarizing each feed's overall performance using multiple descriptive statistics (mean, median, standard deviation, minimum, maximum, and sample count). These profiles were standardized using StandardScaler to put every statistic on a comparable scale, then reduced to a single principal component via PCA, and finally compared pairwise using cosine similarity to generate feed-to-feed recommendations.

FINDINGS
The PCA transformation compressed each feed type's multi-dimensional performance profile down to a single score along one axis, effectively ranking feeds from lowest-performing to highest-performing.

LIMITATION
Reducing the data to a single principal component, as specified in the project requirements, introduces a notable mathematical constraint: cosine similarity between one-dimensional vectors can only take two values +1.0 (same sign, same direction) or -1.0 (opposite sign). This means the model effectively sorts feed types into two broad groups (below-average vs. above-average performers) rather than producing a genuinely graded similarity ranking. Feed types near the midpoint of the distribution are especially sensitive to this limitation. This isn't a flaw in the implementation, it's a consequence of collapsing multiple performance dimensions into a single comparison axis and it's worth being transparent about rather than treating tied results as more precise than they actually are.

BUSINESS IMPLICATIONS AND RECOMMENDATIONS
- Reducing feed performance to a single composite score and using it to group similar products gives the company an automatable, low-maintenance way to suggest substitutes when a preferred feed becomes unavailable or a farmer wants to compare alternatives.
- Feed types with PC1 scores near zero sit at the boundary between performance tiers, and an automated single-recommendation system may group them inconsistently depending on small changes in input data. In a production setting, such cases should be flagged for human review rather than trusted at face value.
- If the business's real product catalog were used instead of this proxy dataset, with more feed types and more performance dimensions, retaining more than one principal component would produce a continuous, more nuanced similarity ranking instead of the binary above/below-average split seen here.